In [1]:
# Stage 1 artifacts 복원
import pickle
from pathlib import Path

artifacts_path = Path('artifacts/stage1_data.pkl')

if not artifacts_path.exists():
    # Drive에서 가져옴
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    import shutil
    artifacts_path.parent.mkdir(exist_ok=True)
    shutil.copy('/content/drive/MyDrive/bakerstreet_artifacts/stage1_data.pkl', artifacts_path)

with open(artifacts_path, 'rb') as f:
    stage1 = pickle.load(f)

cycle_demo = stage1['cycle_demo']
fanin_demo = stage1['fanin_demo']
fanout_demo = stage1['fanout_demo']
icij_samples = stage1['icij_samples']
ACCOUNT_TO_JURISDICTION = stage1['account_to_jurisdiction']

print("Stage 1 artifacts loaded.")
print(f"  IBM accounts: {len(ACCOUNT_TO_JURISDICTION)}")
print(f"  ICIJ samples: {sum(len(v) for v in icij_samples.values())} entities across {len(icij_samples)} jurisdictions")
print(f"  CYCLE: {cycle_demo['tx_count']} tx, FAN-IN: {fanin_demo['tx_count']} tx, FAN-OUT: {fanout_demo['tx_count']} tx")

Mounted at /content/drive
Stage 1 artifacts loaded.
  IBM accounts: 15
  ICIJ samples: 80 entities across 8 jurisdictions
  CYCLE: 6 tx, FAN-IN: 3 tx, FAN-OUT: 4 tx


In [2]:
# IBM 계좌 ↔ ICIJ entity 1:1 픽
used_node_ids = set()

hub_picks = {
    '811C597B0': {  # FAN-IN hub (BVI, receiver only)
        'jurisdiction': 'BVI',
        'condition': lambda r: r['service_provider'] == 'Mossack Fonseca',
        'role_rationale': "FAN-IN hub. Mossack Fonseca client + BVI registration = strongest shell company signal.",
    },
    '80769F1A0': {  # FAN-OUT hub (Delaware, sender only)
        'jurisdiction': 'Delaware',
        'condition': lambda r: 'Fund' in str(r['name']),
        'role_rationale': "FAN-OUT hub. Delaware Fund GP structure naturally distributes capital to multiple LPs.",
    },
}


def find_entity(jurisdiction_short, condition_fn=None):
    pool = icij_samples[jurisdiction_short]
    for _, row in pool.iterrows():
        if row['node_id'] in used_node_ids:
            continue
        if condition_fn is None or condition_fn(row):
            return row
    # fallback: first available regardless of condition
    for _, row in pool.iterrows():
        if row['node_id'] not in used_node_ids:
            return row
    raise ValueError(f"No entity available in {jurisdiction_short}")


ACCOUNT_TO_ENTITY = {}
ACCOUNT_RATIONALE = {}

# Hub picks (curated)
for ibm_acct, spec in hub_picks.items():
    entity = find_entity(spec['jurisdiction'], spec['condition'])
    ACCOUNT_TO_ENTITY[ibm_acct] = entity
    ACCOUNT_RATIONALE[ibm_acct] = spec['role_rationale']
    used_node_ids.add(entity['node_id'])
    print(f"[HUB] {ibm_acct} ({spec['jurisdiction']:9s}) → {entity['name']}")
    print(f"      rationale: {spec['role_rationale']}")
    print()

# Auto-pick remaining 13
for ibm_acct, juris in ACCOUNT_TO_JURISDICTION.items():
    if ibm_acct in ACCOUNT_TO_ENTITY:
        continue
    entity = find_entity(juris)
    ACCOUNT_TO_ENTITY[ibm_acct] = entity
    used_node_ids.add(entity['node_id'])
    print(f"  {ibm_acct} ({juris:9s}) → {entity['name']}")

print(f"\nTotal mapped: {len(ACCOUNT_TO_ENTITY)} / {len(ACCOUNT_TO_JURISDICTION)}")

[HUB] 811C597B0 (BVI      ) → TOUCH INDUSTRIES CO., LIMITED
      rationale: FAN-IN hub. Mossack Fonseca client + BVI registration = strongest shell company signal.

[HUB] 80769F1A0 (Delaware ) → The Peritus Fund GP, LLC
      rationale: FAN-OUT hub. Delaware Fund GP structure naturally distributes capital to multiple LPs.

  803A568D0 (Malta    ) → PATERNA SHIPPING LIMITED
  8012F9500 (Bahamas  ) → Miko Valley Ltd.
  8102CA2C0 (Cyprus   ) → TILIKO LIMITED
  8040FAF80 (BVI      ) → ARCH Secretaries Limited
  80AC28E60 (BVI      ) → Special Situations Global Limited
  80CF37D80 (Panama   ) → QUEZEL MANAGEMENT INC.
  811C599A0 (Cayman   ) → Galaxy Quantitative Fund SPC
  811B83280 (Bahamas  ) → LAKELAND INVESTMENTS S.A.
  812D22980 (Singapore) → DELTA ADVISORY PTE. LTD.
  804529040 (BVI      ) → KOPTECH INVESTMENTS LTD.
  811A9BD50 (Singapore) → The Green Point Trust
  800BFCD30 (BVI      ) → ADAMO PROPERTIES LIMITED
  8020BC410 (Cayman   ) → Desdemona Corp.

Total mapped: 15 / 15


In [3]:
# 셀 3 — shared_attribute 매트릭스 정의

# CYCLE entity 식별 (편의용 단축 변수)
MALTA  = '803A568D0'  # Paterna Shipping (start/end)
BAHAMA = '8012F9500'  # Miko Valley
CYPRUS = '8102CA2C0'  # Tiliko
BVI_1  = '8040FAF80'  # ARCH Secretaries (Yen receiver)
BVI_2  = '80AC28E60'  # Special Situations Global
PANAMA = '80CF37D80'  # Quezel Management (Brazil Real receiver)

# FAN-IN
FANIN_HUB  = '811C597B0'  # Touch Industries (BVI, Mossack Fonseca)
FANIN_S1   = '811C599A0'  # Galaxy Quantitative (Cayman)
FANIN_S2   = '811B83280'  # Lakeland Investments (Bahamas)
FANIN_S3   = '812D22980'  # Delta Advisory (Singapore)

# FAN-OUT
FANOUT_HUB = '80769F1A0'  # Peritus Fund GP (Delaware)
FANOUT_R1  = '804529040'  # Koptech Investments (BVI)
FANOUT_R2  = '811A9BD50'  # Green Point Trust (Singapore)
FANOUT_R3  = '800BFCD30'  # Adamo Properties (BVI)
FANOUT_R4  = '8020BC410'  # Desdemona Corp (Cayman)


# CYCLE shared_attribute matrix
# 각 entry: (entity_a, entity_b, evidence_kind, strength, synth_rationale)
CYCLE_SHARED_ATTRS = [
    # === core 3개 (강) ===
    (MALTA, PANAMA, "shared registered agent",
     "strong",
     "Both registered through Mossack Fonseca, linking cycle start (Paterna) and Brazil Real receiver (Quezel)."),

    (BVI_1, BVI_2, "same registration address",
     "strong",
     "Both BVI entities registered at the same agent's address (typical for Mossack Fonseca BVI clients)."),

    (MALTA, CYPRUS, "matching IP address",
     "strong",
     "Accounting activity from the same IP — Paterna (Malta) and Tiliko (Cyprus), both EU time-zone shells."),

    # === 보조 2개 (약) ===
    (BAHAMA, BVI_1, "common nominee director",
     "weak",
     "A single nominee director listed for both Miko Valley (Bahamas) and ARCH Secretaries (BVI)."),

    (BVI_2, PANAMA, "co-incorporation timing",
     "weak",
     "Special Situations Global and Quezel incorporated within the same week — coordinated setup."),
]


# FAN-IN: hub와 sender 3개 사이에 shared agent
FANIN_SHARED_ATTRS = [
    (FANIN_HUB, FANIN_S1, "shared registered agent", "strong",
     "Touch Industries (hub) and Galaxy Quantitative both Mossack Fonseca clients."),
    (FANIN_HUB, FANIN_S2, "shared registered agent", "strong",
     "Touch Industries (hub) and Lakeland Investments share Mossack Fonseca registration."),
    (FANIN_HUB, FANIN_S3, "shared registered agent", "strong",
     "Touch Industries (hub) and Delta Advisory share Mossack Fonseca registration."),
]


# FAN-OUT: hub와 receiver 4개 사이에 shared beneficial owner
FANOUT_SHARED_ATTRS = [
    (FANOUT_HUB, FANOUT_R1, "shared beneficial owner", "strong",
     "Peritus Fund GP and Koptech share an ultimate beneficial owner."),
    (FANOUT_HUB, FANOUT_R2, "shared beneficial owner", "strong",
     "Peritus Fund GP and Green Point Trust share an ultimate beneficial owner."),
    (FANOUT_HUB, FANOUT_R3, "shared beneficial owner", "strong",
     "Peritus Fund GP and Adamo Properties share an ultimate beneficial owner."),
    (FANOUT_HUB, FANOUT_R4, "shared beneficial owner", "strong",
     "Peritus Fund GP and Desdemona Corp share an ultimate beneficial owner."),
]


ALL_SHARED_ATTRS = CYCLE_SHARED_ATTRS + FANIN_SHARED_ATTRS + FANOUT_SHARED_ATTRS

# 검증 출력
print("=== Shared attribute matrix ===\n")
print(f"CYCLE: {len(CYCLE_SHARED_ATTRS)} edges")
for a, b, kind, strength, _ in CYCLE_SHARED_ATTRS:
    print(f"  [{strength:6s}] {a} ↔ {b}  ({kind})")

print(f"\nFAN-IN: {len(FANIN_SHARED_ATTRS)} edges")
for a, b, kind, strength, _ in FANIN_SHARED_ATTRS:
    print(f"  [{strength:6s}] {a} ↔ {b}  ({kind})")

print(f"\nFAN-OUT: {len(FANOUT_SHARED_ATTRS)} edges")
for a, b, kind, strength, _ in FANOUT_SHARED_ATTRS:
    print(f"  [{strength:6s}] {a} ↔ {b}  ({kind})")

print(f"\nTotal shared_attribute edges: {len(ALL_SHARED_ATTRS)}")

# 각 entity가 최소 1개 shared_attribute에 참여하는지 검증
participating = set()
for a, b, _, _, _ in ALL_SHARED_ATTRS:
    participating.add(a)
    participating.add(b)

all_entities = set(ACCOUNT_TO_ENTITY.keys())
isolated = all_entities - participating
if isolated:
    print(f"\n⚠️  Isolated entities (no shared_attribute): {isolated}")
else:
    print(f"\n✅ All {len(all_entities)} entities participate in at least one shared_attribute")

=== Shared attribute matrix ===

CYCLE: 5 edges
  [strong] 803A568D0 ↔ 80CF37D80  (shared registered agent)
  [strong] 8040FAF80 ↔ 80AC28E60  (same registration address)
  [strong] 803A568D0 ↔ 8102CA2C0  (matching IP address)
  [weak  ] 8012F9500 ↔ 8040FAF80  (common nominee director)
  [weak  ] 80AC28E60 ↔ 80CF37D80  (co-incorporation timing)

FAN-IN: 3 edges
  [strong] 811C597B0 ↔ 811C599A0  (shared registered agent)
  [strong] 811C597B0 ↔ 811B83280  (shared registered agent)
  [strong] 811C597B0 ↔ 812D22980  (shared registered agent)

FAN-OUT: 4 edges
  [strong] 80769F1A0 ↔ 804529040  (shared beneficial owner)
  [strong] 80769F1A0 ↔ 811A9BD50  (shared beneficial owner)
  [strong] 80769F1A0 ↔ 800BFCD30  (shared beneficial owner)
  [strong] 80769F1A0 ↔ 8020BC410  (shared beneficial owner)

Total shared_attribute edges: 12

✅ All 15 entities participate in at least one shared_attribute


In [5]:
# 셀 4 (수정) — RawRecord 빌더 v2

raw_store = {}

# === ALL_SHARED_ATTRS 재정의: cluster_key 추가 ===
# 매트릭스를 셀 3에서 다시 안 짜고, 여기서 enrich

CYCLE_SHARED_ATTRS_V2 = [
    # CYCLE: cluster_key=None → 각 pair마다 별도 record
    (MALTA, PANAMA, "shared registered agent", "strong",
     "Both registered through Mossack Fonseca, linking cycle start (Paterna) and Brazil Real receiver (Quezel).",
     None),  # cluster_key
    (BVI_1, BVI_2, "same registration address", "strong",
     "Both BVI entities registered at the same agent's address.",
     None),
    (MALTA, CYPRUS, "matching IP address", "strong",
     "Accounting activity from the same IP — Paterna (Malta) and Tiliko (Cyprus), both EU time-zone shells.",
     None),
    (BAHAMA, BVI_1, "common nominee director", "weak",
     "A single nominee director listed for both Miko Valley (Bahamas) and ARCH Secretaries (BVI).",
     None),
    (BVI_2, PANAMA, "co-incorporation timing", "weak",
     "Special Situations Global and Quezel incorporated within the same week — coordinated setup.",
     None),
]

FANIN_SHARED_ATTRS_V2 = [
    (FANIN_HUB, FANIN_S1, "shared registered agent", "strong",
     "Touch Industries (hub) and Galaxy Quantitative both Mossack Fonseca clients.",
     "fanin_mossack_cluster"),
    (FANIN_HUB, FANIN_S2, "shared registered agent", "strong",
     "Touch Industries (hub) and Lakeland Investments share Mossack Fonseca registration.",
     "fanin_mossack_cluster"),
    (FANIN_HUB, FANIN_S3, "shared registered agent", "strong",
     "Touch Industries (hub) and Delta Advisory share Mossack Fonseca registration.",
     "fanin_mossack_cluster"),
]

FANOUT_SHARED_ATTRS_V2 = [
    (FANOUT_HUB, FANOUT_R1, "shared beneficial owner", "strong",
     "Peritus Fund GP and Koptech share an ultimate beneficial owner.",
     "fanout_peritus_bo"),
    (FANOUT_HUB, FANOUT_R2, "shared beneficial owner", "strong",
     "Peritus Fund GP and Green Point Trust share an ultimate beneficial owner.",
     "fanout_peritus_bo"),
    (FANOUT_HUB, FANOUT_R3, "shared beneficial owner", "strong",
     "Peritus Fund GP and Adamo Properties share an ultimate beneficial owner.",
     "fanout_peritus_bo"),
    (FANOUT_HUB, FANOUT_R4, "shared beneficial owner", "strong",
     "Peritus Fund GP and Desdemona Corp share an ultimate beneficial owner.",
     "fanout_peritus_bo"),
]

ALL_SHARED_ATTRS_V2 = CYCLE_SHARED_ATTRS_V2 + FANIN_SHARED_ATTRS_V2 + FANOUT_SHARED_ATTRS_V2


# === 빌더 v2 ===

import pandas as pd

def ent_id(ibm_acct):
    return f"ent_{ACCOUNT_TO_ENTITY[ibm_acct]['node_id']}"


def build_transaction_records():
    records = []
    for source_name, demo_data, prefix in [
        ('CYCLE', cycle_demo, 'cycle'),
        ('FAN-IN', fanin_demo, 'fanin'),
        ('FAN-OUT', fanout_demo, 'fanout'),
    ]:
        for i, tx in enumerate(demo_data['transactions']):
            records.append({
                'id': f"tx_{prefix}_{i+1:02d}",
                'type': 'transaction',
                'payload': {
                    'sender_account': tx['from_account'],
                    'receiver_account': tx['to_account'],
                    'sender_bank': tx['from_bank'],
                    'receiver_bank': tx['to_bank'],
                    'amount_received': tx['amount_received'],
                    'receiving_currency': tx['receiving_currency'],
                    'amount_paid': tx['amount_paid'],
                    'payment_currency': tx['payment_currency'],
                    'timestamp': tx['timestamp'],
                    'payment_format': tx['payment_format'],
                    'pattern_source': source_name,
                    'is_laundering': bool(tx['is_laundering']),
                }
            })
    return records


def build_account_records():
    records = []
    for ibm_acct, entity_row in ACCOUNT_TO_ENTITY.items():
        records.append({
            'id': f"acct_{ibm_acct}",
            'type': 'account',
            'payload': {
                'ibm_account_id': ibm_acct,
                'holder_entity_id': ent_id(ibm_acct),
                'holder_metadata': {
                    'icij_node_id': int(entity_row['node_id']),
                    'name': entity_row['name'],
                    'jurisdiction': ACCOUNT_TO_JURISDICTION[ibm_acct],
                    'jurisdiction_full': entity_row['jurisdiction_description'],
                    'service_provider': entity_row['service_provider'] if pd.notna(entity_row['service_provider']) else None,
                    'incorporation_date': entity_row['incorporation_date'] if pd.notna(entity_row['incorporation_date']) else None,
                    'address': entity_row['address'] if pd.notna(entity_row['address']) else None,
                    'sourceID': entity_row['sourceID'],
                },
                'role_rationale': ACCOUNT_RATIONALE.get(ibm_acct),
            }
        })
    return records


def build_registration_and_ip_records():
    """
    cluster_key가 있으면 묶고, 없으면 pair별 별도 record.
    matching IP는 ip type, 나머지는 registration type.
    """
    records = []

    # cluster_key가 있는 항목 → 누적
    cluster_records = {}  # (kind, cluster_key) → record

    # cluster_key가 None인 항목 → 즉시 record 생성
    pair_counter = {'agent': 0, 'address': 0, 'nominee': 0, 'incorp': 0, 'ip': 0, 'bo': 0}

    def kind_to_prefix_and_type(kind):
        # (record_id_prefix, RawRecord type)
        return {
            "shared registered agent":   ('agent',   'registration'),
            "same registration address": ('address', 'registration'),
            "common nominee director":   ('nominee', 'registration'),
            "co-incorporation timing":   ('incorp',  'registration'),
            "shared beneficial owner":   ('bo',      'registration'),
            "matching IP address":       ('ip',      'ip'),
        }[kind]

    def kind_to_payload_template(kind):
        # type별 payload 기본값
        return {
            "shared registered agent": {
                'kind': 'registered_agent',
                'agent_name': 'Mossack Fonseca',
            },
            "same registration address": {
                'kind': 'shared_address',
                'address_name': 'Mossack Fonseca BVI Office, Akara Building, Road Town, Tortola',
            },
            "common nominee director": {
                'kind': 'nominee_director',
                'nominee_name': 'Anonymous nominee',
            },
            "co-incorporation timing": {
                'kind': 'co_incorporation',
                'incorporation_window': '2006-W12 (March 20-26, 2006)',
            },
            "shared beneficial owner": {
                'kind': 'beneficial_owner',
                'owner_name': 'Anonymous BO',
            },
            "matching IP address": {
                'ip_address': '203.0.113.42',  # RFC 5737 documentation range
            },
        }[kind]

    for a, b, kind, strength, rationale, cluster_key in ALL_SHARED_ATTRS_V2:
        ent_a = ent_id(a)
        ent_b = ent_id(b)
        prefix, rec_type = kind_to_prefix_and_type(kind)

        if cluster_key is not None:
            # cluster 단위 묶음
            key = (kind, cluster_key)
            if key not in cluster_records:
                pair_counter[prefix] += 1
                rid = f"{'reg' if rec_type == 'registration' else 'ip_log'}_{prefix}_{pair_counter[prefix]:03d}"
                cluster_records[key] = {
                    'id': rid,
                    'type': rec_type,
                    'payload': {
                        **kind_to_payload_template(kind),
                        'clients' if rec_type == 'registration' else 'users': set(),
                        'evidence_for_pairs': [],
                    }
                }
            participants_key = 'clients' if rec_type == 'registration' else 'users'
            cluster_records[key]['payload'][participants_key].update([ent_a, ent_b])
            cluster_records[key]['payload']['evidence_for_pairs'].append({
                'a': ent_a, 'b': ent_b, 'rationale': rationale, 'strength': strength,
            })
        else:
            # pair-level 별도 record
            pair_counter[prefix] += 1
            rid = f"{'reg' if rec_type == 'registration' else 'ip_log'}_{prefix}_{pair_counter[prefix]:03d}"
            participants_key = 'clients' if rec_type == 'registration' else 'users'
            records.append({
                'id': rid,
                'type': rec_type,
                'payload': {
                    **kind_to_payload_template(kind),
                    participants_key: [ent_a, ent_b],
                    'evidence_for_pair': {'a': ent_a, 'b': ent_b, 'rationale': rationale, 'strength': strength},
                }
            })

    # cluster records 후처리: set → list
    for record in cluster_records.values():
        if 'clients' in record['payload']:
            record['payload']['clients'] = sorted(record['payload']['clients'])
        if 'users' in record['payload']:
            record['payload']['users'] = sorted(record['payload']['users'])
        records.append(record)

    return records


# === 실행 ===
raw_store = {}

for r in (build_transaction_records()
          + build_account_records()
          + build_registration_and_ip_records()):
    raw_store[r['id']] = r

# === 결과 ===
type_counts = {}
for rid, rec in raw_store.items():
    type_counts[rec['type']] = type_counts.get(rec['type'], 0) + 1

print("=== RawRecord build v2 ===")
for t, c in sorted(type_counts.items()):
    print(f"  {t:15s}: {c}")
print(f"  TOTAL: {len(raw_store)}")

# 모든 record id 나열
print("\n=== Record IDs ===")
for rid in sorted(raw_store.keys()):
    rec = raw_store[rid]
    if rec['type'] == 'registration':
        clients = rec['payload'].get('clients', [])
        print(f"  {rid:25s} ({rec['type']:12s}) clients: {len(clients)}")
    elif rec['type'] == 'ip':
        users = rec['payload'].get('users', [])
        print(f"  {rid:25s} ({rec['type']:12s}) users: {len(users)}")
    else:
        print(f"  {rid:25s} ({rec['type']:12s})")

# CYCLE Malta-Panama agent record 확인
print("\n=== reg_agent_001 (CYCLE Malta-Panama, should be pair-only) ===")
import json
print(json.dumps(raw_store['reg_agent_001'], indent=2, ensure_ascii=False, default=str))

# FAN-IN cluster record 확인
print("\n=== reg_agent_002 (FAN-IN cluster, should have 4 clients) ===")
print(json.dumps(raw_store['reg_agent_002'], indent=2, ensure_ascii=False, default=str))

=== RawRecord build v2 ===
  account        : 15
  ip             : 1
  registration   : 6
  transaction    : 13
  TOTAL: 35

=== Record IDs ===
  acct_800BFCD30            (account     )
  acct_8012F9500            (account     )
  acct_8020BC410            (account     )
  acct_803A568D0            (account     )
  acct_8040FAF80            (account     )
  acct_804529040            (account     )
  acct_80769F1A0            (account     )
  acct_80AC28E60            (account     )
  acct_80CF37D80            (account     )
  acct_8102CA2C0            (account     )
  acct_811A9BD50            (account     )
  acct_811B83280            (account     )
  acct_811C597B0            (account     )
  acct_811C599A0            (account     )
  acct_812D22980            (account     )
  ip_log_ip_001             (ip          ) users: 2
  reg_address_001           (registration) clients: 2
  reg_agent_001             (registration) clients: 2
  reg_agent_002             (registration) clients

In [6]:
# 셀 5 — Self-test: shared_attribute의 검증 회로 점검

print("=" * 70)
print("SELF-TEST: Verification loop integrity")
print("=" * 70)

# evidence kind → 그 evidence의 raw record를 찾는 함수
def find_supporting_record(ent_a, ent_b, kind, raw_store):
    """
    ent_a와 ent_b가 모두 clients/users에 포함되며 kind와 일치하는 record를 찾음.
    type-specific 매칭:
    - "shared registered agent" → registration with kind=registered_agent
    - "same registration address" → registration with kind=shared_address
    - "common nominee director" → registration with kind=nominee_director
    - "co-incorporation timing" → registration with kind=co_incorporation
    - "shared beneficial owner" → registration with kind=beneficial_owner
    - "matching IP address" → ip
    """
    target_payload_kind = {
        "shared registered agent":   ("registration", "registered_agent"),
        "same registration address": ("registration", "shared_address"),
        "common nominee director":   ("registration", "nominee_director"),
        "co-incorporation timing":   ("registration", "co_incorporation"),
        "shared beneficial owner":   ("registration", "beneficial_owner"),
        "matching IP address":       ("ip", None),
    }
    rec_type, payload_kind = target_payload_kind[kind]

    matches = []
    for rid, rec in raw_store.items():
        if rec['type'] != rec_type:
            continue
        if payload_kind is not None and rec['payload'].get('kind') != payload_kind:
            continue
        participants_key = 'clients' if rec_type == 'registration' else 'users'
        participants = rec['payload'].get(participants_key, [])
        if ent_a in participants and ent_b in participants:
            matches.append(rid)
    return matches


# === Test 1: shared_attribute 검증 ===
print("\n--- Test 1: every shared_attribute edge resolves to a raw record ---")
test1_failures = []
for a, b, kind, strength, rationale, cluster_key in ALL_SHARED_ATTRS_V2:
    ent_a = ent_id(a)
    ent_b = ent_id(b)
    matches = find_supporting_record(ent_a, ent_b, kind, raw_store)
    if len(matches) == 0:
        test1_failures.append((a, b, kind, "no record found"))
        print(f"  ❌ {a}↔{b} ({kind}): NO supporting record")
    else:
        print(f"  ✅ {a}↔{b} ({kind:30s}) → {matches[0]}")

# === Test 2: flow edge 검증 ===
print("\n--- Test 2: every flow transaction has a tx record with matching sender/receiver ---")
test2_failures = []

def collect_flow_edges():
    """CYCLE/FAN-IN/FAN-OUT의 모든 transaction을 flow edge로 변환."""
    edges = []
    for source_name, demo_data, prefix in [
        ('CYCLE', cycle_demo, 'cycle'),
        ('FAN-IN', fanin_demo, 'fanin'),
        ('FAN-OUT', fanout_demo, 'fanout'),
    ]:
        for i, tx in enumerate(demo_data['transactions']):
            edges.append({
                'tx_id': f"tx_{prefix}_{i+1:02d}",
                'sender_account': tx['from_account'],
                'receiver_account': tx['to_account'],
            })
    return edges


for edge in collect_flow_edges():
    rec = raw_store.get(edge['tx_id'])
    if rec is None:
        test2_failures.append((edge['tx_id'], "missing record"))
        print(f"  ❌ {edge['tx_id']}: missing record")
        continue
    if (rec['payload']['sender_account'] != edge['sender_account']
        or rec['payload']['receiver_account'] != edge['receiver_account']):
        test2_failures.append((edge['tx_id'], "sender/receiver mismatch"))
        print(f"  ❌ {edge['tx_id']}: sender/receiver mismatch")
        continue
    print(f"  ✅ {edge['tx_id']}: {edge['sender_account']} → {edge['receiver_account']}")


# === Test 3: 모든 entity가 최소 1개 account record에 holder로 등록됨 ===
print("\n--- Test 3: every entity has an account record ---")
test3_failures = []
all_entity_ids = {ent_id(acct) for acct in ACCOUNT_TO_ENTITY.keys()}
holder_in_acct = set()
for rid, rec in raw_store.items():
    if rec['type'] == 'account':
        holder_in_acct.add(rec['payload']['holder_entity_id'])

missing = all_entity_ids - holder_in_acct
if missing:
    test3_failures.append(missing)
    print(f"  ❌ Entities without account record: {missing}")
else:
    print(f"  ✅ All {len(all_entity_ids)} entities have an account record")


# === Test 4: 핵심 시나리오 시뮬레이션 ===
# "사용자가 cycle Malta-Panama edge의 'shared registered agent' 태그를 클릭"
# → raw_store에서 reg_agent_001을 찾아 client 목록을 보여줌
# → 사용자가 그 record가 진짜로 두 entity를 연결하는지 직접 검증 가능
print("\n--- Test 4: scenario walkthrough (Malta↔Panama agent click) ---")
malta_ent = ent_id(MALTA)
panama_ent = ent_id(PANAMA)
matches = find_supporting_record(malta_ent, panama_ent, "shared registered agent", raw_store)
if matches:
    rid = matches[0]
    rec = raw_store[rid]
    print(f"  Click on edge: {malta_ent} ↔ {panama_ent} (shared registered agent)")
    print(f"  → RawPanel opens record: {rid}")
    print(f"  → Agent: {rec['payload']['agent_name']}")
    print(f"  → Clients listed: {rec['payload']['clients']}")
    print(f"  → Both entities present in clients: "
          f"{malta_ent in rec['payload']['clients'] and panama_ent in rec['payload']['clients']}")
    print(f"  ✅ User can independently verify the claim against this record")
else:
    print(f"  ❌ No record supports this evidence — verification loop broken")


# === 요약 ===
print("\n" + "=" * 70)
total_failures = len(test1_failures) + len(test2_failures) + len(test3_failures)
if total_failures == 0:
    print("🎉 ALL TESTS PASSED — verification loop is fully grounded")
    print(f"   {len(ALL_SHARED_ATTRS_V2)} shared_attribute edges, "
          f"{len(collect_flow_edges())} flow edges, "
          f"{len(all_entity_ids)} entities — all reverse-traceable")
else:
    print(f"⚠️  {total_failures} failures — see above")
print("=" * 70)

SELF-TEST: Verification loop integrity

--- Test 1: every shared_attribute edge resolves to a raw record ---
  ✅ 803A568D0↔80CF37D80 (shared registered agent       ) → reg_agent_001
  ✅ 8040FAF80↔80AC28E60 (same registration address     ) → reg_address_001
  ✅ 803A568D0↔8102CA2C0 (matching IP address           ) → ip_log_ip_001
  ✅ 8012F9500↔8040FAF80 (common nominee director       ) → reg_nominee_001
  ✅ 80AC28E60↔80CF37D80 (co-incorporation timing       ) → reg_incorp_001
  ✅ 811C597B0↔811C599A0 (shared registered agent       ) → reg_agent_002
  ✅ 811C597B0↔811B83280 (shared registered agent       ) → reg_agent_002
  ✅ 811C597B0↔812D22980 (shared registered agent       ) → reg_agent_002
  ✅ 80769F1A0↔804529040 (shared beneficial owner       ) → reg_bo_001
  ✅ 80769F1A0↔811A9BD50 (shared beneficial owner       ) → reg_bo_001
  ✅ 80769F1A0↔800BFCD30 (shared beneficial owner       ) → reg_bo_001
  ✅ 80769F1A0↔8020BC410 (shared beneficial owner       ) → reg_bo_001

--- Test 2: every flo

In [7]:
# 셀 6 — Stage 2 artifacts 직렬화 + Drive 미러링

import json, pickle
from pathlib import Path

Path('artifacts').mkdir(exist_ok=True)

# Stage 2 산출물 묶음
stage2 = {
    'raw_store': raw_store,
    'shared_attrs': ALL_SHARED_ATTRS_V2,
    'account_to_entity_node_id': {
        ibm_acct: int(entity_row['node_id'])
        for ibm_acct, entity_row in ACCOUNT_TO_ENTITY.items()
    },
    'account_rationale': ACCOUNT_RATIONALE,

    # Stage 1 키 정보도 같이 (Stage 3에서 한 파일만 로드하면 되도록)
    'cycle_demo': cycle_demo,
    'fanin_demo': fanin_demo,
    'fanout_demo': fanout_demo,
    'account_to_jurisdiction': ACCOUNT_TO_JURISDICTION,
}

# === pickle (Python objects 그대로) ===
with open('artifacts/stage2_data.pkl', 'wb') as f:
    pickle.dump(stage2, f)

print(f"Saved: artifacts/stage2_data.pkl "
      f"({Path('artifacts/stage2_data.pkl').stat().st_size/1024:.1f} KB)")

# === JSON (사람이 읽기 + git 커밋 / 프론트엔드 직접 사용) ===
# tuple은 JSON 표현 불가 → list로 변환
def to_json_safe(obj):
    if isinstance(obj, tuple):
        return list(obj)
    if isinstance(obj, dict):
        return {k: to_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_json_safe(v) for v in obj]
    return obj

stage2_json = to_json_safe(stage2)

with open('artifacts/stage2_data.json', 'w', encoding='utf-8') as f:
    json.dump(stage2_json, f, indent=2, ensure_ascii=False, default=str)

print(f"Saved: artifacts/stage2_data.json "
      f"({Path('artifacts/stage2_data.json').stat().st_size/1024:.1f} KB)")

# === 프론트엔드용 raw_store만 따로 (synthetic_dataset.json) ===
# 프론트엔드가 fetch할 형식: nodes + rawStore
synthetic_dataset = {
    'meta': {
        'source': 'IBM AML-Data (HI-Small) + ICIJ Offshore Leaks',
        'patterns': ['CYCLE', 'FAN-IN', 'FAN-OUT'],
        'entity_count': 15,
        'flow_edge_count': 13,
        'shared_attribute_edge_count': 12,
    },
    'nodes': [
        {
            'id': f"ent_{entity_row['node_id']}",
            'label': entity_row['name'],
            'jurisdiction': ACCOUNT_TO_JURISDICTION[ibm_acct],
            'jurisdiction_full': entity_row['jurisdiction_description'],
            'type': 'entity',
            'ibm_account_id': ibm_acct,
        }
        for ibm_acct, entity_row in ACCOUNT_TO_ENTITY.items()
    ],
    'rawStore': raw_store,

    # 데모용 보조: 미리 만들어진 edge 후보 (Pattern Detector 산출물 시뮬레이션)
    'preDetectedEdges': {
        'flow': [
            {'tx_id': r['id'],
             'from': f"ent_{ACCOUNT_TO_ENTITY[r['payload']['sender_account']]['node_id']}",
             'to': f"ent_{ACCOUNT_TO_ENTITY[r['payload']['receiver_account']]['node_id']}",
             'pattern_source': r['payload']['pattern_source']}
            for r in raw_store.values() if r['type'] == 'transaction'
        ],
        'shared_attribute': [
            {'from': f"ent_{ACCOUNT_TO_ENTITY[a]['node_id']}",
             'to': f"ent_{ACCOUNT_TO_ENTITY[b]['node_id']}",
             'kind': kind,
             'strength': strength}
            for a, b, kind, strength, _, _ in ALL_SHARED_ATTRS_V2
        ],
    },
}

with open('artifacts/synthetic_dataset.json', 'w', encoding='utf-8') as f:
    json.dump(synthetic_dataset, f, indent=2, ensure_ascii=False, default=str)

print(f"Saved: artifacts/synthetic_dataset.json "
      f"({Path('artifacts/synthetic_dataset.json').stat().st_size/1024:.1f} KB)")

# === Drive 미러 ===
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import shutil
drive_dir = Path('/content/drive/MyDrive/bakerstreet_artifacts')
drive_dir.mkdir(exist_ok=True)

for fname in ['stage2_data.pkl', 'stage2_data.json', 'synthetic_dataset.json']:
    shutil.copy(f'artifacts/{fname}', drive_dir)

print(f"\n=== Drive mirror: {drive_dir} ===")
for f in sorted(drive_dir.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size/1024:.1f} KB)")

print("\n🎉 Stage 2 complete.")

Saved: artifacts/stage2_data.pkl (11.1 KB)
Saved: artifacts/stage2_data.json (31.4 KB)
Saved: artifacts/synthetic_dataset.json (28.6 KB)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

=== Drive mirror: /content/drive/MyDrive/bakerstreet_artifacts ===
  stage1_data.json  (34.8 KB)
  stage1_data.pkl  (13.9 KB)
  stage2_data.json  (31.4 KB)
  stage2_data.pkl  (11.1 KB)
  synthetic_dataset.json  (28.6 KB)

🎉 Stage 2 complete.
